# 🧪 高级专家 机考模拟练习三：A/B 实验进阶与核心统计推断专场

> **🎯 考核目标**：针对 A/B 实验的深水区进行高强度巩固。涵盖：SRM（多维下钻）、MDE反推、Delta Method（比值型指标）、以及各类显著性检验的选型。

---

## 🧪 试题一：SRM 的进阶排查 (Simpson's Paradox 预警)

**📝 你的任务**：
一个实验大盘分流（50:50）数据显示：实验组 50,500 人，对照组 49,500 人。
用大盘单维度卡方检验算出来，这个差异的 p-value 可能在 0.001 边缘徘徊。
但作为高级分析师，你怀疑是**某个特定维度**（比如操作系统）导致的局部漏水。

你会如何设计一套自动化的代码逻辑，遍历不同的维度，找到真正发生 SRM 的具体分层？

In [19]:
import pandas as pd
from scipy.stats import chisquare

data = pd.DataFrame({
    'os': ['iOS', 'iOS', 'Android', 'Android', 'Windows', 'Windows'],
    'group': ['exp', 'ctrl', 'exp', 'ctrl', 'exp', 'ctrl'],
    'users': [20000, 20100, 30000, 28000, 500, 1400] # 看看哪个群组的分流崩了
})

# ==================
# 你的代码写在这里：

# 1. 遍历 data 里的 distinct 'os'
# 2. 分别计算该 OS 下的 卡方 p_value (期望人数 = 实验组和对照组人数之和 / 2)
# 3. 打印出哪个操作系统的分流存在明显的 SRM 异常（p_value < 0.001）
data_grouped = data.pivot_table(
    index = 'os',
    columns = 'group',
    values='users',
    aggfunc='sum'
)
for x in data_grouped.index:
    exp_users = data_grouped.loc[x]['exp']
    ctrl_users = data_grouped.loc[x]['ctrl']
    total_users = exp_users + ctrl_users
    
    observed = [ctrl_users,exp_users]
    expected = [total_users / 2,total_users / 2]
    
    diff = exp_users - ctrl_users
    
    if chisquare(observed, expected)[1] < 0.001:
        print(f'{x} 下的分流存在明显的 SRM 异常,p_value={chisquare(observed, expected)[1]:.6f}')
        print(f'ctrl:exp={int(round(ctrl_users * 100 / total_users,0))}:{int(round(exp_users * 100 / total_users,0))}, diff={diff}')


# ==================

# ---------------------
# 💡 面试官防守要点：能看到 Windows 的实验组:对照组 比例是 500:1400，Android是 30000:28000，这些局部的失衡汇聚到大盘上偶尔会被稀释。所以 SRM 必须分维交叉下钻监控！

Android 下的分流存在明显的 SRM 异常,p_value=0.000000
ctrl:exp=48:52, diff=2000
Windows 下的分流存在明显的 SRM 异常,p_value=0.000000
ctrl:exp=74:26, diff=-900


## 🧪 试题二：显著性检验选型 (均值类 vs 比例类 vs 极其长尾分布)

**📝 你的任务**：
作为面试官，我给你准备了 3 组不同特征的实验数据。请你为其选择**最合适**的假设检验方法，并计算 P-value。

1. `metric_a`: 用户的次日留存（0或1，比例类）
2. `metric_b`: 用户的完播率（0%～100% 的连续值，近似正态分布，均值类）
3. `metric_c`: 用户的打赏金额（极其符合二八定律，严重右偏，非正态长尾）

In [29]:
import numpy as np
import scipy.stats as stats
from statsmodels.stats.proportion import proportions_ztest

np.random.seed(42)
# 1. 留存 (二项分布)
retention_exp = np.random.binomial(1, 0.45, 1000)
retention_ctrl = np.random.binomial(1, 0.40, 1000)

# 2. 完播率 (连续, 近似正态)
finish_rate_exp = np.random.normal(0.6, 0.1, 1000)
finish_rate_ctrl = np.random.normal(0.58, 0.1, 1000)

# 3. 打赏金额 (严重右偏长尾，对数正态模拟)
tip_exp = np.random.lognormal(mean=1.5, sigma=2, size=1000)
tip_ctrl = np.random.lognormal(mean=1.3, sigma=2, size=1000)

# ==================
# 你的代码写在这里：

# 1. 检验 metric_a (留存，比例类) -> 选择什么检验？(提示：proportions_ztest 或 卡方)
stat,p_value = stats.ttest_ind(retention_exp,retention_ctrl)
print(f'metric_a T检验 p_value:{p_value:.4f},{'显著' if p_value <0.05 else '不显著'}')

count = [sum(retention_exp),sum(retention_ctrl)]
nobs = [len(retention_exp),len(retention_ctrl)]
stat,p_value = proportions_ztest(count,nobs)
print(f'metric_a Z检验 p_value:{p_value:.4f},{'显著' if p_value <0.05 else '不显著'}')

# 2. 检验 metric_b (完播率，均值类且大样本) -> 选择什么检验？(提示：stats.ttest_ind)
stat,p_value = stats.ttest_ind(finish_rate_exp,finish_rate_ctrl)
print(f'metric_b T检验 p_value:{p_value:.4f},{'显著' if p_value <0.05 else '不显著'}')

# 3. 检验 metric_c (打赏金额，极端长尾) -> 选择什么检验？(提示：可以尝试非参数检验 stats.mannwhitneyu)
stat,p_value = stats.mannwhitneyu(tip_exp,tip_ctrl)
print(f'metric_c Mann-Whitney U检验 p_value:{p_value:.4f},{'显著' if p_value <0.05 else '不显著'}')

import numpy as np
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

# 假设你的原始数据是 tip_exp 和 tip_ctrl
# ==========================================

# 💡 Part 1: 广度检验 —— 转化率 (是否打赏)
# 将所有大于0的值变为1，等于0的值保持为0
is_tipped_exp = (tip_exp > 0).astype(int)
is_tipped_ctrl = (tip_ctrl > 0).astype(int)

# 统计打赏人数和总人数
count_tipped = [is_tipped_exp.sum(), is_tipped_ctrl.sum()]
nobs = [len(is_tipped_exp), len(is_tipped_ctrl)]

# 因为样本>1000，使用 Z检验 (proportions_ztest)
stat_cv, p_val_cv = proportions_ztest(count_tipped, nobs)
print(f"✅ Part 1 - 打赏转化率检验:")
print(f"   实验组打赏率: {is_tipped_exp.mean():.2%}, 对照组: {is_tipped_ctrl.mean():.2%}")
print(f"   Z检验 p_value: {p_val_cv:.4f} ({'显著' if p_val_cv < 0.05 else '不显著'})")
print("-" * 40)

# 💡 Part 2: 深度检验 —— 客单价 (仅针对打赏过的用户)
# 这一步过滤掉所有 0 元的白嫖用户，只提取真实产生利润的买单用户
amount_exp_nonzero = tip_exp[tip_exp > 0]
amount_ctrl_nonzero = tip_ctrl[tip_ctrl > 0]

# 因为剔除 0 之后，数据更接近普通的连续变量分布（虽然可能仍有长尾，但不至于被海量0稀释）
# 使用普通的 T检验 (Welch's t-test，设置 equal_var=False 防御方差不齐)
stat_amt, p_val_amt = stats.ttest_ind(amount_exp_nonzero, amount_ctrl_nonzero, equal_var=False)

print(f"✅ Part 2 - 打赏客单价检验 (排除0元用户):")
print(f"   实验组均价: ￥{amount_exp_nonzero.mean():.2f}, 对照组均价: ￥{amount_ctrl_nonzero.mean():.2f}")
print(f"   T检验 p_value: {p_val_amt:.4f} ({'显著' if p_val_amt < 0.05 else '不显著'})")
print("=" * 40)

# 💡 终极分析结论 (供向老板汇报用)：
# 1. 如果 Part 1 显著，Part 2 不显著: "策略成功破圈，吸引了更多新客尝试打赏，但大家的平均消费力没变。"
# 2. 如果 Part 1 不显著，Part 2 显著: "策略没能拉新，但成功刺激了核心忠诚用户的消费潜能（薅羊毛老用户的钱）。"
# 3. 如果都不显著: "策略彻底无效。"

# ==================

# ---------------------
# 💡 面试官防守要点：对于极度长尾的打赏金额，直接用 T 检验完全可能失效（方差太大导致检验效能极低）。应该考虑：1. 对数变换后 T 检验 2. 排除离群值 3. 分阶段模型(先看是否打赏，然后过滤出打赏的人来分析打赏金额) 4. 非参数检验 (Mann-Whitney U test)。




metric_a T检验 p_value:0.6200,不显著
metric_a Z检验 p_value:0.6198,不显著
metric_b T检验 p_value:0.0000,显著
metric_c Mann-Whitney U检验 p_value:0.1046,不显著
✅ Part 1 - 打赏转化率检验:
   实验组打赏率: 100.00%, 对照组: 100.00%
   Z检验 p_value: nan (不显著)
----------------------------------------
✅ Part 2 - 打赏客单价检验 (排除0元用户):
   实验组均价: ￥27.43, 对照组均价: ￥26.76
   T检验 p_value: 0.9078 (不显著)


./.pyenv/versions/3.12.12/lib/python3.12/site-packages/statsmodels/stats/weightstats.py:792: RuntimeWarning: invalid value encountered in scalar divide
  zstat = value / std


## 🧪 试题三：比值型指标 (Ratio Metric) 的方差推导与 Delta Method

**📝 你的任务**：
在电商实验中，“人均转化金额”（总 GMV / 总访问天数）是一个典型的比值型指标（Ratio Metric）。它的分子（GMV）和分母（访问天数）都是用户的属性，且通常高度相关。
对于比值型指标 $R = \frac{X}{Y}$，你不能直接把个体的 `X/Y` 求方差，那是错的。

请根据 Delta Method 泰勒展开的一阶近似公式，估算总体比值 $R$ 的方差：
$\text{Var}(\frac{X}{Y}) \approx \frac{1}{\mu_Y^2} \text{Var}(X) + \frac{\mu_X^2}{\mu_Y^4} \text{Var}(Y) - 2 \frac{\mu_X}{\mu_Y^3} \text{Cov}(X, Y)$

给定一组实验大盘中的用户数据：

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
y = np.random.poisson(lam=5, size=5000) # y: 平均访问5天
x = y * 20 + np.random.normal(0, 50, 5000) # x: 消费额。购买金额和访问天数高度相关
df_ratio = pd.DataFrame({'spend_x': x, 'days_y': y})

# ==================
# 你的代码写在这里：

# 1. 计算均值 mu_x, mu_y
mean_x = x.mean()
mean_y = y.mean()

# 2. 计算方差 var_x, var_y 和 协方差 cov_xy (提示：可以使用 np.cov() 来拿协方差矩阵)
# 注意: var_x 也可以用 df_ratio['spend_x'].var()
var_x = x.var()
var_y = y.var()
cov_xy = np.cov(x,y)[0,1]
# 3. 实现 Delta Method 估计 Ratio 的方差 var_ratio
var_ratio = 1/mean_y**2 * var_x + mean_x **2 / (mean_y **4)*var_y - 2 * mean_x/mean_y**3 * cov_xy
# ==================
print(f'var_ratio={var_ratio:.4f}')

# ---------------------
# 💡 架构师级标准答案：如果是真的想要估算比值指标的增量，在实际工业界中不仅用到 Delta Method，现在更流行的方式是用 CUPED + Delta Method，或者 Bootstrap 重抽样。但 Delta Method 的手写是必考项。直接算 (X/Y).var() 会被直接淘汰！

var_ratio=96.4854


,spend_x,days_y
0,79.011514,5
1,46.159463,4
2,89.963611,4
3,102.789481,5
4,153.197792,5


## 🧪 试题四：MDE 的反解 - 我们到底能察觉多小的变化？

**📝 你的任务**：
在测试准备阶段，你的老板问你：“咱们这次只有 10,000 的 DAU 可以进实验（实验组和对照组各拿 5,000 人），你觉得在统计学规律下，我们能测出多大的 MDE？”

此时核心指标是转化率，**基线转化率** $p = 0.20$。
已知：比例型样本量公式为 $n = (Z_{\alpha/2} + Z_{\beta})^2 \times \frac{2 \times p(1-p)}{mde^2}$
假设 Alpha=0.05 ($Z \approx 1.96$), Beta=0.20 ($Z \approx 0.84$)。

请用 Python 代码写方程反解出 MDE，并以百分比的形式打印输出。

In [37]:
import math

# 参数定义
z_alpha_2 = 1.96
z_beta = 0.84
p_baseline = 0.20
n_per_group = 5000

# ==================
# 你的代码写在这里：

# 1. 根据公式反解 mde 的平方 (mde_squared)
mde_square = (z_alpha_2 + z_beta)**2 * 2*p_baseline*(1-p_baseline)/n_per_group
# 2. 开根号得到 mde 绝对值
mde = math.sqrt(mde_square)
# 3. 打印输出绝对提升百分数。
print(f'MDE:{mde:.4f},绝对提升{mde*100:.2f}')
print(f'含义：转化率从 {p_baseline:.0%} 到 {p_baseline + mde:.2%}，在当前样本量下才能被检测出显著。')
# ==================

# ---------------------
# 💡 面试官防守要点：反解出来的 MDE 约为 0.0224 (即绝对提升 2.24%)。这意味着，如果转化率从 20% 只提升到 21%，在目前的样本量配置下是测不出统计显著的 (p-value 会 > 0.05，即 Power 不足)。老板听到这个结论后，要么决定增加全站推全的流量比例延长时长，要么放弃这个微小的优化改版。

MDE:0.0224,绝对提升2.24
含义：转化率从 20% 到 22.24%，在当前样本量下才能被检测出显著。


## 🧪 试题五：CUPED 再进阶 - OLS 多元回归视角

**📝 你的任务**：
你在之前练习用协方差公式 $\theta = \frac{Cov(Y, X)}{Var(X)}$ 手算了单个协变量的 CUPED。
其实，CUPED 的本质就是**线性回归（OLS）**的残差。
$Y_{cuped} = Y - \hat{\beta}_X \times (X - E[X])$ 的 $\hat{\beta}_X$ 正是独立变量（X）在多元或者一元线性回归（OLS回归）时的斜率系数。

这次，我们引入了**多个协变量**（例如：实验前 14 天的购买额 `pre_value_1`，实验前 14 天的点击次数 `pre_clicks_2`）。
请使用 `statsmodels.api` 来做 OLS 多元拟合，一次性提取多个 $\theta$ 权重并生成修正后的新多维目标指标。

In [39]:
import statsmodels.api as sm
import pandas as pd
import numpy as np

np.random.seed(42)
pre_val = np.random.normal(500, 100, 5000)
pre_clicks = np.random.poisson(20, 5000)
# 实验期购买额和前面两个都有关
post_val = pre_val * 1.05 + pre_clicks * 2.5 + np.random.normal(0, 50, 5000) 

df_ab2 = pd.DataFrame({'pre_value_1': pre_val, 'pre_clicks_2': pre_clicks, 'post_value': post_val})

# ==================
# 你的代码写在这里：

# 1. 构造特征矩阵 X (包含截距项)
# 提示: X = sm.add_constant(df_ab2[['pre_value_1', 'pre_clicks_2']])
# Y = df_ab2['post_value']

# 2. 训练 OLS 模型 (model = sm.OLS(Y, X).fit())

# 3. 提取特征的回归系数 theta1 (对应 pre_value_1), theta2 (对应 pre_clicks_2)

# 4. 生成多变量组合的 CUPED 降噪指标： 
# post_value_cuped = Y - theta1 * (X1 - E[X1]) - theta2 * (X2 - E[X2])

# ==================

# ---------------------
# 💡 架构师级标准答案：利用 OLS / 随机森林乃至 XGBoost 拟合出 Y_hat 作为我们的先验预测（机器学习扩展版 CV, Control Variates），能极大限度榨取历史数据来降噪指标从而缩短所需的实验时间。这也是顶级大厂做 A/B 实验平台的核心高级算法！

# 1. 构造特征矩阵 X (包含截距项)
X = sm.add_constant(df_ab2[['pre_value_1', 'pre_clicks_2']])
Y = df_ab2['post_value']

# 2. 训练 OLS 模型
model = sm.OLS(Y, X).fit()

# 3. 提取特征的回归系数 theta1, theta2
theta1 = model.params['pre_value_1']
theta2 = model.params['pre_clicks_2']
print(f"theta1 (购买额权重): {theta1:.4f}")
print(f"theta2 (点击数权重): {theta2:.4f}")

# 4. 生成多变量组合的 CUPED 降噪指标
# Y_cuped = Y - theta1 * (X1 - E[X1]) - theta2 * (X2 - E[X2])
mean_x1 = df_ab2['pre_value_1'].mean()
mean_x2 = df_ab2['pre_clicks_2'].mean()
df_ab2['post_value_cuped'] = (
    Y 
    - theta1 * (df_ab2['pre_value_1'] - mean_x1) 
    - theta2 * (df_ab2['pre_clicks_2'] - mean_x2)
)

# 验证降噪效果
var_before = Y.var()
var_after = df_ab2['post_value_cuped'].var()
print(f"\n降噪前方差: {var_before:.2f}")
print(f"降噪后方差: {var_after:.2f}")
print(f"方差缩减: {(1 - var_after/var_before)*100:.1f}%")



theta1 (购买额权重): 1.0509
theta2 (点击数权重): 2.5793

降噪前方差: 13530.45
降噪后方差: 2494.57
方差缩减: 81.6%
